In [1]:
from gitsource import GithubRepositoryDataReader

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [3]:
files = reader.read()

In [4]:
documents = []

In [5]:
for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
print(documents[0])

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [8]:
len(documents)

72

# Indexing and Searching

## Index the documents with minsearch - make 'content' a text field and 'filename' a keyword field.

In [17]:
from minsearch import Index

In [10]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

In [11]:
index.fit(documents)

## Then search with this query

In [12]:
query = "How does the agentic loop keep calling the model until it stops ?"

In [13]:
response = index.search(query)

In [14]:
[doc['filename'] for doc in response]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md',
 '01-agentic-rag/lessons/12-rag-revision.md',
 '01-agentic-rag/lessons/01-intro.md',
 '05-monitoring/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md',
 '05-monitoring/lessons/02-assistant-setup.md']

In [9]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
import os

In [10]:
load_dotenv()

True

In [11]:
#openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [18]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [19]:
response = assistant.rag(query=query)

In [20]:
print(response)

('It keeps calling the model inside a `while True` loop. After each model response, the code checks whether there were any `function_call` items.\n\n- If there was a function call, it runs the tool, appends the tool result to `messages`, and loops again.\n- If there were no function calls, it breaks out of the loop.\n\nSo the stop condition is simply: **no function calls in the latest response**.\n\nIn the lesson’s code, that is controlled by the `has_function_calls` flag:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nThat’s how the loop keeps going until the model is done.', ResponseUsage(input_tokens=7121, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=139, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7260))


In [21]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60
    
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        'input_cost': input_cost,
        'output_cost': output_cost,
        'total_cost': total_cost
    }

In [22]:
calculate_gpt54mini_price(
    response[1].input_tokens,
    response[1].output_tokens
)

{'input_cost': 0.00106815,
 'output_cost': 8.34e-05,
 'total_cost': 0.0011515499999999999}

In [12]:
from gitsource import chunk_documents

In [13]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
print(chunks[0])

{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone

In [15]:
len(chunks)

295

In [18]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

In [19]:
index.fit(chunks)

In [37]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="llama-3.1-8b-instant"
)

In [38]:
response = assistant.rag(query=query)

In [39]:
response

("The agentic loop keeps calling the model until it stops because of two main reasons:\n\n1. **Conditional Loop Break**: The loop stops when there are no more function calls in the model's response. The condition for breaking the loop is `if has_function_calls == False`. When the model's response does not contain any function calls, the loop breaks, and the model's final answer is returned.\n\n2. **Model's Ability to Stop**: The model itself is designed to stop making function calls when it has reached a final answer or when there's no more information to be gathered. This ability to self-regulate the conversation is a key feature of the agentic loop.\n\nHere's a step-by-step explanation of how the loop stops:\n\n1.  The model generates a response, and the loop checks if there are any function calls in the response.\n2.  If there are function calls, the loop processes them, executes the tools, and updates the messages list.\n3.  The loop then checks again if there are any function call

## Turning it into an agent

In [23]:
def search(query):
    """ Search in the knowledge base the query from a course student """
    boost_dict = {"content": 3.0, "filename": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [29]:
instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

In [22]:
query = "How does the agentic loop work, and how is it different from plain RAG?"

In [28]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback, OpenAIResponsesRunner

In [25]:
agent_tool = Tools()

In [26]:
agent_tool.add_tool(search)

In [27]:
agent_tool.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search in the knowledge base the query from a course student',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [32]:
runner = OpenAIResponsesRunner(
    tools=agent_tool,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [33]:
results = runner.loop(
    prompt=query,
    callback=callback
)

-> Response received


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-KTESiEo08biaXsdbGdHICZky on tokens per min (TPM): Limit 100000, Used 100000, Requested 8545. Please try again in 61h31m26.4s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}